In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json
from functools import partial
import numpy as np
import pandas as pd
import torch
from fancy_einsum import einsum

from transformers import AutoModelForCausalLM
from datasets import load_from_disk


import transformer_lens as tl
from circuitsvis.attention import attention_heads
import transformer_lens.utils as utils
from transformer_lens import ActivationCache, HookedTransformer, HookedTransformerConfig

from geomechinterp.causal.mygpt import SymbolTokenizer, DataCollator
from geomechinterp.tflens.utils import load_gpt2_to_hooked_transformer
from geomechinterp.causal.base_functions import all_binary_generators_func_names
from geomechinterp.causal.utils import DisplayChain

In [5]:
model_hf = AutoModelForCausalLM.from_pretrained("../gpt_rope_custom/checkpoint-407000/")


hook_config = HookedTransformerConfig(d_vocab=29, n_ctx=128, d_model=128,
                                      d_head=32, n_layers=6, n_heads=4,
                                      rotary_dim=64, act_fn='gelu_new', 
                                      original_architecture='GPT2LMHeadModel')

model = load_gpt2_to_hooked_transformer(model_hf, hook_config)

# model.tokenizer = SymbolTokenizer()

model_hf.to("mps").eval()
model.to("mps").eval()
print('')

Moving model to device:  mps



In [6]:
input_ids = torch.tensor([1, 2, 3, 3, 1, 2, 5, 6, 2, 10, 2, 3, 15, 18, 20, 7]).to("mps")

out1 = model_hf(input_ids)
out2, hooked_act = model.run_with_cache(input_ids)

k = 1
print('WARNING: SMALL DIFFERENCES IN ACTIVATIONS ARE ACCUMULATING, PROBABLY DUE TO DISREPANCIES IN THE LAYER NORM!')
print('---'*10)
for k in range(out2.shape[1]):
    # use cosine similarity after softmax
    print(torch.cosine_similarity(torch.nn.functional.softmax(out1['logits'][k,:], dim=-1), torch.nn.functional.softmax(out2[:,k,:], dim=-1)))    


------------------------------
tensor([0.9994], device='mps:0', grad_fn=<SumBackward1>)
tensor([1.], device='mps:0', grad_fn=<SumBackward1>)
tensor([1.], device='mps:0', grad_fn=<SumBackward1>)
tensor([1.0000], device='mps:0', grad_fn=<SumBackward1>)
tensor([1.], device='mps:0', grad_fn=<SumBackward1>)
tensor([1.], device='mps:0', grad_fn=<SumBackward1>)
tensor([0.9523], device='mps:0', grad_fn=<SumBackward1>)
tensor([1.], device='mps:0', grad_fn=<SumBackward1>)
tensor([0.9994], device='mps:0', grad_fn=<SumBackward1>)
tensor([0.9347], device='mps:0', grad_fn=<SumBackward1>)
tensor([1.], device='mps:0', grad_fn=<SumBackward1>)
tensor([1.], device='mps:0', grad_fn=<SumBackward1>)
tensor([0.9226], device='mps:0', grad_fn=<SumBackward1>)
tensor([0.9750], device='mps:0', grad_fn=<SumBackward1>)
tensor([0.9235], device='mps:0', grad_fn=<SumBackward1>)
tensor([0.9390], device='mps:0', grad_fn=<SumBackward1>)


In [7]:
tokenizer = SymbolTokenizer()
test_dataset = load_from_disk("../tokenized_test")
data_collator = DataCollator(test_dataset, device="mps")  

In [8]:
idx = -9
test_case = test_dataset[idx]
out = model_hf(torch.tensor(test_case['input_ids']).to("mps"))
decoded_str = tokenizer.decode(out['logits'].argmax(dim=-1).tolist())

print(DisplayChain.from_json(test_case['generator'], base_functions=all_binary_generators_func_names))
print(test_case['text'][6:])
print(decoded_str[5:])

Chain of functions:
IndependentFeature(function=plus_minus_f, global_control=None) (stochastic)
DependentFeature(function=ab_f, controls=[+-, ab_prev, position_parity], truth_table=[0, 1, 0, 1, 1, 0, 0, 0])
DependentFeature(function=case_f, controls=[+-, ab, position_parity], truth_table=[1, 0, 0, 0, 1, 0, 1, 1])
-A +a +b -B -A -B +b +a -B -B +b +a -B +a -B -B +b +a +b -B -A +a -B +a +b +a +b +a
-A -a +A +B +A +B -A -a +A +B +b +a +B +a +B +B +b +a +b +B +A +a +


In [9]:
# select subset of deterministic patterns
deterministic_test_dataset = []
for case in test_dataset:
    if case['stochastic']:
        continue
    else:
        deterministic_test_dataset.append(case)

print(len(deterministic_test_dataset))

46


In [10]:
torch.tensor(test_case['input_ids']).to("mps")

tensor([6, 2, 4, 6, 3, 4, 6, 2, 4, 5, 0, 4, 5, 1, 4, 6, 3, 4, 6, 2, 4, 6, 3, 4,
        5, 1, 4, 5, 0, 4, 6, 3, 4, 6, 3, 4, 5, 1, 4, 5, 0, 4, 6, 3, 4, 5, 0, 4,
        6, 3, 4, 6, 3, 4, 5, 1, 4, 5, 0, 4, 5, 1, 4, 6, 3, 4, 6, 2, 4, 5, 0, 4],
       device='mps:0')

In [20]:
idx = 2
test_case = deterministic_test_dataset[idx]
out = model(torch.tensor(test_case['input_ids']).to("mps"))
decoded_str = tokenizer.decode(out.argmax(dim=-1).tolist())[0]

print(DisplayChain.from_json(test_case['generator'], base_functions=all_binary_generators_func_names))
print(test_case['text'][6:])
print(decoded_str[5:])

Chain of functions:
DependentFeature(function=plus_minus_f, controls=[ab_prev, position_parity], truth_table=[1, 0, 1, 1])
DependentFeature(function=ab_f, controls=[+-, ab_prev, position_parity], truth_table=[1, 0, 1, 1, 1, 1, 1, 0])
DependentFeature(function=case_f, controls=[+-, ab, ab_prev], truth_table=[1, 0, 0, 0, 0, 0, 0, 1])
+b +b +a -a +b +b +a -a +b +b +a -a +b +b +a -a +b +b +a -a +b +b +a -a +b +b +a -a
+a +a -a +a +b -b -b +b +b -b -a +b +a +b +a +b +a +b +a +a +b +b -


In [22]:
idx = 34
test_case = deterministic_test_dataset[idx]
out = model(torch.tensor(test_case['input_ids']))
decoded_str = tokenizer.decode(out.argmax(dim=-1).tolist())[0]

print(DisplayChain.from_json(test_case['generator'], base_functions=all_binary_generators_func_names))
print(test_case['text'][6:])
print(decoded_str[5:])

Chain of functions:
DependentFeature(function=plus_minus_f, controls=[position_parity], truth_table=[0, 1])
DependentFeature(function=ab_f, controls=[+-, ab_prev, position_parity], truth_table=[1, 0, 0, 1, 0, 1, 0, 1])
DependentFeature(function=case_f, controls=[+-, ab_prev, position_parity], truth_table=[0, 1, 0, 0, 0, 1, 1, 0])
+a -b +A -a +a -b +A -a +a -b +A -a +a -b +A -a +a -b +A -a +a -b +A -a +a -b +A -a
+A +a +A +a +a +b +a +b +a -b +a -a +a -b +A -a +a -b +A -a +a -b -


#### Visualize MyGPT Activations in Reduced Dimensions

#### MyGPT Activation Patching and Circuit Analysis

In [23]:
from IPython.display import HTML, IFrame
from tqdm import tqdm

import plotly.express as px
import plotly.graph_objects as go
from matplotlib import pyplot as plt
import plotly.io as pio

In [153]:
from geomechinterp.utils import run_model_on_pattern, run_model_on_pattern_and_plot

In [ ]:
pattern = 'a a a a a a a a a a a a a a a a a a a a a a a a a'
loss = run_model_on_pattern(model, pattern)
print(loss)

In [ ]:
pattern = 'b b b b b b b b b b b b b b b b b b b b b b b b b'
_ = run_model_on_pattern_and_plot(model, pattern)

In [ ]:
pattern = 'a b a b a b a b a b a b a b a b a b a b a b a b a b'
_ = run_model_on_pattern_and_plot(model, pattern, annotate_tokens=True)

In [ ]:
pattern = '+ - + - + - + - + - + - + - + - + - + - + - + - + -'
_ = run_model_on_pattern_and_plot(model, pattern, annotate_tokens=True)

In [ ]:
ALL_DETERMINISTIC_PATTERNS_LIST = list(ALL_DETERMINISTIC_PATTERNS_AND_GENERATORS2.keys())

losses = []
for pattern in ALL_DETERMINISTIC_PATTERNS_LIST:
    loss = run_model_on_pattern(model, pattern[2:], exclude_first_k=5)
    losses.append(loss)

df = pd.DataFrame({'pattern': [p[:29] for p in ALL_DETERMINISTIC_PATTERNS_LIST], 'loss': losses})

In [ ]:
df.sort_values(by='loss', ascending=False)

In [ ]:
ALL_DETERMINISTIC_PATTERNS_LIST

In [ ]:
pattern = '+B -a -a +A +B -a -a +A +B -a -a +A +B -a -a +A +B -a -a +A'
print(ALL_DETERMINISTIC_PATTERNS_AND_GENERATORS2[pattern]['generator'])
_ = run_model_on_pattern_and_plot(model, pattern, annotate_tokens=True)

In [ ]:
# check that all symbols in patters match to a single token
symbols = ['a','b', '+', '-', 'A', 'B', '1', '2', '>', '<', '?', '!', '[', ']', '(', ')']
for symbol in symbols:
    if symbol not in model.tokenizer.get_vocab():
        print(f"Symbol {symbol} not found in tokens")

In [ ]:

# Function to tokenize patterns, run them through the model, and measure uncertainty
def run_model_on_pattern_and_plot(model, pattern, device):
    # Preprocess pattern
    tokens = pattern.replace("...", "").split()  # Split the pattern into individual tokens
    token_ids = model.to_tokens(tokens).to(device)  # Convert tokens to token ids
    token_ids = token_ids[:, :512]  # Limit the sequence length if necessary

    # Forward pass through the model
    logits, cache = model(token_ids, return_type="logits")
    
    # Calculate uncertainty (entropy over logits)
    softmax_logits = torch.nn.functional.softmax(logits, dim=-1)
    entropy = -torch.sum(softmax_logits * torch.log(softmax_logits + 1e-8), dim=-1)

    # Return uncertainty (logit entropy) for each step
    return entropy.cpu().detach().numpy()

# Run model on each pattern and register uncertainty
uncertainties = {}
for pattern in patterns:
    uncertainties[pattern] = run_model_on_pattern_and_plot(model, pattern, device)

# Display the uncertainties for each pattern
for pattern, uncertainty in uncertainties.items():
    print(f"Pattern: {pattern}\nUncertainty at each step:\n{uncertainty}\n")